In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :exponential

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [exponential_model] Fitting chain 2 (tau=34)
[ Info: [exponential] iter 1000/1000000 elapsed=3.9s, rate=0.155, mean=[1.406, 0.00121, 1.385, 0.235], std=[0.1018, 0.000373, 0.1471, 0.1368] [ADAPT]
[ Info: [exponential] iter 2000/1000000 elapsed=6.9s, rate=0.132, mean=[1.589, 0.00116, 1.427, 0.161], std=[0.1868, 0.000282, 0.1123, 0.1172] [ADAPT]
[ Info: [exponential] iter 3000/1000000 elapsed=9.1s, rate=0.121, mean=[1.749, 0.00109, 1.426, 0.137], std=[0.2563, 0.000257, 0.0919, 0.1006] [ADAPT]
[ Info: [exponential] iter 4000/1000000 elapsed=11.3s, rate=0.112, mean=[1.833, 0.00105, 1.424, 0.123], std=[0.2575, 0.000234, 0.0798, 0.0897] [ADAPT]
[ Info: [exponential] iter 5000/1000000 elapsed=13.4s, rate=0.109, mean=[1.883, 0.00103, 1.427, 0.115], std=[0.2477, 0.000218, 0.0722, 0.0815] [ADAPT]
[ Info: [exponential] iter 6000/1000000 elapsed=15.6s, rate=0.105, mean=[1.912, 0.00102, 1.443, 0.110], std=[0.2332, 0.000204, 0.0733, 0.0751] [ADAPT]
[ Info: [exponential] iter 7000/1000000 elap